# 03 — Graph Building & Analisis SNA
**Tujuan:** Membangun Aspect Co-occurrence Network dan menganalisis strukturnya menggunakan metrik Social Network Analysis.

---

In [ ]:
import os
import sys
# Auto-adjust CWD to project root if executed from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sys
sys.path.insert(0, 'src')
from graph_builder import build_cooccurrence_graph, compute_centrality, detect_communities, export_gexf
from utils import summarize_graph

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

with open('data/processed/absa_results.json', encoding='utf-8') as f:
    absa_results = json.load(f)

print(f'Total entri ABSA: {len(absa_results)}')

## 1. Build Graph

In [ ]:
G = build_cooccurrence_graph(absa_results)
summary = summarize_graph(G)

print('Statistik Graph:')
for k, v in summary.items():
    print(f'  {k:15s}: {v}')

print()
print('Nodes :', list(G.nodes()))
print('Edges :', [(u,v,d['weight']) for u,v,d in G.edges(data=True)])

## 2. Centrality Analysis

In [ ]:
centrality = compute_centrality(G)

df_centrality = pd.DataFrame(centrality).rename(columns={
    'degree': 'Degree', 'betweenness': 'Betweenness', 'eigenvector': 'Eigenvector'
}).round(4)
df_centrality.index.name = 'Aspek'
df_centrality = df_centrality.sort_values('Degree', ascending=False)

print('Centrality Metrics:')
display(df_centrality)

# Simpan ke CSV
import os
os.makedirs('../results', exist_ok=True)
df_centrality.to_csv('results/centrality_metrics.csv')
print('\nTersimpan di results/centrality_metrics.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['Degree', 'Betweenness', 'Eigenvector']
colors  = ['#5C6BC0', '#26A69A', '#EF5350']

for ax, metric, color in zip(axes, metrics, colors):
    vals = df_centrality[metric].sort_values()
    bars = ax.barh(vals.index, vals.values, color=color, edgecolor='white')
    for b in bars:
        ax.text(b.get_width()+0.005, b.get_y()+b.get_height()/2,
                f'{b.get_width():.3f}', va='center', fontsize=9)
    ax.set_title(f'{metric} Centrality', fontweight='bold')
    ax.set_xlim(0, max(vals.values) * 1.2)

plt.suptitle('Perbandingan Metrik Centrality Antar Aspek', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/centrality_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Community Detection

In [ ]:
partition = detect_communities(G)
n_communities = len(set(partition.values()))
print(f'Jumlah komunitas terdeteksi: {n_communities}')
print()
for comm_id in sorted(set(partition.values())):
    members = [n for n, c in partition.items() if c == comm_id]
    print(f'  Komunitas {comm_id}: {members}')

## 4. Visualisasi Graph

In [ ]:
import matplotlib.colors as mcolors

pos = nx.spring_layout(G, k=0.8, seed=42)
deg_centrality = centrality['degree']

tab_colors = list(mcolors.TABLEAU_COLORS.values())
node_colors = [tab_colors[partition[n] % len(tab_colors)] for n in G.nodes()]
node_sizes  = [3500 * deg_centrality[n] + 1000 for n in G.nodes()]

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights)
edge_widths = [1 + 6 * (w / max_w) for w in edge_weights]
edge_labels = {(u, v): G[u][v]['weight'] for u, v in G.edges()}

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                       alpha=0.85, edgecolors='white', linewidths=2, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5, edge_color='#9E9E9E', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=13, font_weight='bold', ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9, ax=ax)

ax.set_title('Aspect Co-occurrence Network\n(Ukuran node = Degree Centrality, Lebar edge = Frekuensi)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('results/figures/aspect_network.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Edge Weight & Neg-Alignment

In [ ]:
edges_data = []
for u, v, d in G.edges(data=True):
    edges_data.append({
        'Pasangan': f'{u} — {v}',
        'Co-occurrence': d.get('weight', 0),
        'Neg Count': d.get('neg_alignment', 0),
        'Neg Ratio': d.get('neg_ratio', 0)
    })

df_edges = pd.DataFrame(edges_data).sort_values('Co-occurrence', ascending=False)
print('Edge Attributes:')
display(df_edges)

# Simpan
df_edges.to_csv('results/edge_attributes.csv', index=False)
print('\nTersimpan di results/edge_attributes.csv')

In [ ]:
# Export GEXF untuk Gephi
export_gexf(G, path='results/graph_exports/aspect_network.gexf')